# 🚀 AeroGuard TSLM — Notebook 3: Model Architecture & Inference Pre-Check

This notebook tests loading the trained **AeroGuard TSLM** multimodal model, verifying the **TimeSeriesPatchEncoder**, running a forward pass on held-out test telemetry, and comparing the scalar RUL head output against the ground truth.

### Objectives:
1. Verify checkpoint files in `models/aeroguard_tslm/`.
2. Instantiate `AeroGuardTSLM` (`SmolLM-135M-Instruct` backbone + LoRA + patch encoder).
3. Run a forward pass on a held-out test engine window (e.g. Engine #84).
4. Evaluate both the auxiliary scalar RUL prediction and the generated Chain-of-Thought diagnostics.
5. Test physical fault isolation logic (`training.fault_isolation`).

In [ ]:
import sys
from pathlib import Path
import torch

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from training.dataset_loader import CMAPSSCoTDataset, NormalizationStats
from training.train_opentslm import AeroGuardTSLM
from training.fault_isolation import isolate_component_fault

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✔ Using compute device: {device}")

## 1. Verify Checkpoint Artifacts

Check whether the model weights and adapter files are available in `models/aeroguard_tslm/`.

In [ ]:
MODEL_DIR = PROJECT_ROOT / "models" / "aeroguard_tslm"

required_files = [
    MODEL_DIR / "preprocessing.json",
    MODEL_DIR / "tslm_adapters.pt",
    MODEL_DIR / "lora_adapters" / "adapter_config.json",
    MODEL_DIR / "lora_adapters" / "adapter_model.safetensors",
]

for p in required_files:
    exists = p.exists()
    status = "✔ Found" if exists else "❌ Missing"
    print(f"{status}: {p.relative_to(PROJECT_ROOT)}")

## 2. Load Model & Trained Weights

Initialize the multimodal architecture and load the fine-tuned LoRA adapters and patch encoder.

In [ ]:
print("Initializing AeroGuard TSLM architecture...")
model = AeroGuardTSLM(
    model_id="HuggingFaceTB/SmolLM-135M-Instruct",
    device=device
)

# Load patch encoder & RUL head weights
tslm_path = MODEL_DIR / "tslm_adapters.pt"
if tslm_path.is_file():
    state_dict = torch.load(tslm_path, map_location=device)
    model.ts_encoder.load_state_dict(state_dict["patch_encoder"])
    model.rul_head.load_state_dict(state_dict["rul_head"])
    print("✔ Loaded TimeSeriesPatchEncoder & RUL Head weights.")

# Load LoRA adapters for SmolLM
lora_dir = MODEL_DIR / "lora_adapters"
if (lora_dir / "adapter_model.safetensors").is_file():
    model.llm.load_adapter(str(lora_dir), adapter_name="default")
    print("✔ Loaded PEFT LoRA adapters.")

model.to(device)
model.eval()
print("✔ Model ready for inference.")

## 3. Load Held-Out Test Window (Engine #84)

Let's load a 30-cycle telemetry window from strictly held-out **Engine Unit #84** (unseen during training) near failure threshold.

In [ ]:
WINDOWS_PATH = PROJECT_ROOT / "data" / "processed" / "windows.jsonl"
NORM_PATH = MODEL_DIR / "preprocessing.json"

test_dataset = CMAPSSCoTDataset(
    jsonl_path=str(WINDOWS_PATH),
    split="test",
    normalization=str(NORM_PATH) if NORM_PATH.is_file() else None,
    include_targets=True
)

# Find a near-failure window for Engine #84
sample = None
for i in range(len(test_dataset)):
    item = test_dataset[i]
    if item["unit_number"] == 84 and item["rul"] <= 20:
        sample = item
        break

assert sample is not None, "Sample window not found."
print(f"• Selected Window:    {sample['record_id']}")
print(f"• Engine Unit:        #{sample['unit_number']}")
print(f"• Inspection Cycle:   {sample['cycle']}")
print(f"• True Remaining RUL: {sample['rul']} cycles")

## 4. Run Multimodal Inference (RUL & CoT Generation)

Now pass the normalized $[14, 30]$ sensor tensor into AeroGuard TSLM to predict scalar RUL and generate natural language diagnostics.

In [ ]:
sensor_tensor = sample["sensor_series"].to(device) # Shape: [14, 30]

# Generate assessment
pred_rul, diagnostic_text = model.generate_assessment(
    sensor_series=sensor_tensor,
    prompt_text=sample["prompt"],
    max_new_tokens=128
)

print("=" * 65)
print("✈️ AEROGUARD TSLM INFERENCE RESULT")
print("=" * 65)
print(f"• True Remaining Useful Life:      {sample['rul']} cycles")
print(f"• Predicted Remaining Useful Life: {pred_rul:.1f} cycles")
print(f"• Absolute Error:                  {abs(pred_rul - sample['rul']):.1f} cycles")
print("=" * 65)
print("\nGenerated Chain-of-Thought Diagnostics:")
print("-" * 65)
print(diagnostic_text)

## 5. Physical Component Fault Isolation

Test the deterministic fault isolation module to pinpoint the Line-Replaceable Unit (LRU) part and AMM task card based on aerothermal drift.

In [ ]:
# Extract raw window for Engine #84 to compute drifts
raw_series = sample["raw_series"] # Dict of sensor lists
drifts = {k: v[-1] - v[0] for k, v in raw_series.items()}

fault_report = isolate_component_fault(
    t50_drift=drifts.get("sensor_4", 0.0),
    ps30_drift=drifts.get("sensor_11", 0.0),
    bpr_drift=drifts.get("sensor_15", 0.0),
    rul=int(round(pred_rul))
)

print("=== Physical Fault Isolation Report ===")
print(f"• Primary Failing Station: {fault_report.station_name} (Station {fault_report.station_id})")
print(f"• Sub-component Part:      {fault_report.subsystem}")
print(f"• OEM Part Number:         {fault_report.part_number}")
print(f"• Maintenance Task Card:   {fault_report.task_card}")
print(f"• Severity Level:          {fault_report.severity}")
print(f"• MRO Action Directive:    {fault_report.action_directive}")